# Phase 2C.3 - Context Depth 3 (Colab)

Run the locked P2 prompt at depth 3 on the chunking winner selected by Phase 2C.2. Reuse the frozen Phase 2B subsets and Phase 2C.1 retrieval traces; depth 5 is reused from generation screening. A CPU Colab runtime is sufficient.

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='19a2da4429da2989973777013be77e9b7378a7af'
ARM_ID='c3_hierarchical'
CONTEXT_DEPTH=3
RUN_MODE='smoke'                 # smoke | screening
EXECUTE_API_CALLS=False
RETRIEVAL_RESULTS_PATH='/content/drive/MyDrive/newsqa_phase2c/phase2c_retrieval_screening_results.zip'

PHASE2C_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2c-indexes-v1'
PHASE2C_REVISION='09421ba33e75f2cef01dd01451ce4523214589ca'
PHASE2C_FILENAME='phase2c_chunking_indexes_v1.zip'
PHASE2C_SHA256='96030993b38de4d62828ce3d9945ae8fc73647c7c8fd0155d6eb223d0e6a0c90'
PREP_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
PREP_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
PREP_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
PREP_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'
GENERATOR_MODEL='gemini-3.1-flash-lite'; GENERATOR_REASONING='minimal'; GENERATOR_MAX_TOKENS=512; GENERATOR_INTERVAL=4.2
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'; JUDGE_REASONING='low'; JUDGE_MAX_TOKENS=2048
GEMINI_SECRET_NAME='GEMINI_API_KEY_1'; SEED=42; TOP_K=20; RERANK_TOP_N=5
GENERATOR_INPUT_USD_PER_MILLION=0.25; GENERATOR_OUTPUT_USD_PER_MILLION=1.50; JUDGE_INPUT_USD_PER_MILLION=0.15; JUDGE_OUTPUT_USD_PER_MILLION=0.50
ROOT=Path('/content'); PROJECT_ROOT=ROOT/'Text-Mining---NewsQA-RAG'; WORK=ROOT/f'phase2c_{ARM_ID}_d{CONTEXT_DEPTH}_{RUN_MODE}'
DATA=WORK/'data'; RUNTIME=WORK/'runtime'; RUN_DIR=WORK/'run'; RESULTS=WORK/'results'; LOGS=WORK/'logs'

## 1. Setup and immutable inputs

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
from google.colab import drive,userdata
drive.mount('/content/drive')
for path in [DATA,RUNTIME,RUN_DIR,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
assert ARM_ID in {'c1_sentence','c2_paragraph','c3_hierarchical'} and RUN_MODE in {'smoke','screening'}
assert not REPO_COMMIT.startswith('SET_TO_'),'Pin REPO_COMMIT after committing these notebooks'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import pandas as pd,yaml
from huggingface_hub import hf_hub_download
def secret(name):
    try: return userdata.get(name) or ''
    except Exception: return ''
HF_TOKEN=secret('HF_TOKEN'); GENERATOR_KEY=secret(GEMINI_SECRET_NAME); JUDGE_KEY=secret('FIREWORKS_API_KEY')
if EXECUTE_API_CALLS: assert GENERATOR_KEY and JUDGE_KEY,f'Configure {GEMINI_SECRET_NAME} and FIREWORKS_API_KEY'
assert HF_TOKEN,'HF_TOKEN is required for the private preparation repository'
os.environ.update({'PYTHONPATH':str(PROJECT_ROOT/'common'),'PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false','TOKENIZERS_PARALLELISM':'false'})
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def rows(path):
    with Path(path).open(encoding='utf-8') as f: return [json.loads(line) for line in f if line.strip()]
def stable_hash(value): return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def extract(archive,target,marker,prefix=None):
    target.mkdir(parents=True,exist_ok=True)
    if not (target/marker).exists():
        with zipfile.ZipFile(archive) as z:
            members=[item for item in z.infolist() if prefix is None or item.filename.startswith(prefix)]
            z.extractall(target,members)
    assert (target/marker).exists()
p2c_zip=Path(hf_hub_download(repo_id=PHASE2C_REPO_ID,repo_type='dataset',revision=PHASE2C_REVISION,filename=PHASE2C_FILENAME)); assert sha(p2c_zip)==PHASE2C_SHA256
prep_zip=Path(hf_hub_download(repo_id=PREP_REPO_ID,repo_type='dataset',revision=PREP_REVISION,filename=PREP_FILENAME,token=HF_TOKEN)); assert sha(prep_zip)==PREP_SHA256
P2C=DATA/'phase2c'; PREP=DATA/'preparation'; RETRIEVAL=DATA/'retrieval'; folder={'c1_sentence':'sentence','c2_paragraph':'paragraph','c3_hierarchical':'hierarchical'}[ARM_ID]
extract(p2c_zip,P2C,f'{folder}/artifact_manifest.json',f'{folder}/'); extract(prep_zip,PREP,'question_ids/screening.json')
retrieval_zip=Path(RETRIEVAL_RESULTS_PATH); assert retrieval_zip.exists(),'Place the Phase 2C.1 screening result ZIP at RETRIEVAL_RESULTS_PATH'
extract(retrieval_zip,RETRIEVAL,'run_manifest.json')
source_manifest=json.loads((RETRIEVAL/'run_manifest.json').read_text()); assert source_manifest['stage']=='phase2c_retrieval_screening' and source_manifest['run_mode']=='screening'
eligibility=json.loads((RETRIEVAL/'phase2c_retrieval_screening_eligibility.json').read_text()); assert eligibility['selection_permitted']
if RUN_MODE=='screening': assert eligibility['eligibility'][ARM_ID]['eligible'],f'{ARM_ID} was rejected by retrieval screening'
active_ids=json.loads((PREP/f'question_ids/{RUN_MODE}.json').read_text()); judge_ids=json.loads((PREP/f"question_ids/{'smoke' if RUN_MODE=='smoke' else 'judge_calibration'}.json").read_text())
assert set(judge_ids)<=set(active_ids); IDS=RUNTIME/'ids.json'; JUDGE_IDS=RUNTIME/'judge_ids.json'; IDS.write_text(json.dumps(active_ids)); JUDGE_IDS.write_text(json.dumps(judge_ids))
print('Arm:',ARM_ID,'| depth:',CONTEXT_DEPTH,'| questions:',len(active_ids),'| judged:',len(judge_ids))

## 2. Bind the selected arm and frozen retrieval trace

In [ ]:
ARM_ROOT=P2C/folder; chunks=ARM_ROOT/'chunks.jsonl'; testset_source=ARM_ROOT/'testset_resolved.jsonl'; sparse_index=ARM_ROOT/'bge_m3_sparse.pkl'
source_retrievals=RETRIEVAL/f'runs/{ARM_ID}/retrievals.jsonl'; assert source_retrievals.exists(),f'Retrieval bundle does not contain {ARM_ID}'
source_trace_manifest=json.loads((RETRIEVAL/f'runs/{ARM_ID}/run_manifest.json').read_text()); assert source_trace_manifest['inputs'].get('rerank_candidate_n',source_trace_manifest['inputs']['rerank_top_n'])==TOP_K,'C3 retrieval bundle predates parent backfill; rerun notebook 15b'
test_rows=rows(testset_source); selected={r['question_id']:r for r in test_rows if r['question_id'] in set(active_ids)}; assert len(selected)==len(active_ids)
if ARM_ID=='c3_hierarchical':
    parent_by_id={r['id']:r for r in rows(ARM_ROOT/'parents.jsonl')}; child_parent={r['child_id']:r['parent_id'] for r in rows(ARM_ROOT/'child_parent_map.jsonl')}
    transformed_test=[]
    for row in test_rows:
        row=dict(row); row['relevant_chunk_ids']=list(dict.fromkeys(child_parent[cid] for cid in row['relevant_chunk_ids'])); transformed_test.append(row)
    testset=RUNTIME/'testset_resolved_parent.jsonl'
    with testset.open('w',encoding='utf-8') as f:
        for row in transformed_test: f.write(json.dumps(row)+'\n')
    if str(PROJECT_ROOT/'common') not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/'common'))
    from newsqa_rag.retrieval.hierarchical import expand_ranked_children_to_parents
    expanded=RUNTIME/'retrievals_parent.jsonl'
    with expanded.open('w',encoding='utf-8') as f:
        for record in rows(source_retrievals):
            record=dict(record); trace=dict(record['trace']); reranked_child_count=len(trace['reranked_chunks']); trace['retrieved_chunks']=expand_ranked_children_to_parents(trace['retrieved_chunks'],child_parent,parent_by_id,TOP_K); trace['reranked_chunks']=expand_ranked_children_to_parents(trace['reranked_chunks'],child_parent,parent_by_id,RERANK_TOP_N); trace['retrieved_ids']=[x['id'] for x in trace['reranked_chunks']]; trace['contexts']=[x['text'] for x in trace['reranked_chunks']]; trace['hierarchical_expansion']={'reranked_child_count':reranked_child_count,'delivered_parent_count':len(trace['reranked_chunks'])}; record['trace']=trace; f.write(json.dumps(record)+'\n')
    expanded_records={row['question_id']:row for row in rows(expanded)}; short={qid:expanded_records[qid]['trace']['hierarchical_expansion']['delivered_parent_count'] for qid in active_ids if expanded_records[qid]['trace']['hierarchical_expansion']['delivered_parent_count']<RERANK_TOP_N}
    assert not short,f'C3 source trace does not backfill five parents: {list(short.items())[:5]}'
    source_retrievals=expanded
else: testset=testset_source
config=yaml.safe_load((ARM_ROOT/'config.yaml').read_text()); config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING}); config_path=RUNTIME/'config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False))
variant=json.loads((ARM_ROOT/'variant.json').read_text()); variant['pipeline'].update({'config_path':str(config_path),'config_sha256':stable_hash(config)}); variant['artifacts']['chunks']={'path':str(chunks),'sha256':sha(chunks)}; variant['artifacts']['bm25']={'path':str(sparse_index),'sha256':sha(sparse_index)}; variant['artifacts']['testset_resolved']={'path':str(testset),'sha256':sha(testset)}; profile=RUNTIME/'variant.json'; profile.write_text(json.dumps(variant,indent=2,sort_keys=True)+'\n')
prompt=PREP/'prompts/p2.txt'; assert prompt.exists(); print('Source trace:',source_retrievals)

## 3. Resumable generation, judge, and scoring

In [ ]:
def run(command,label,extra_env=None):
    log=LOGS/f'{label}.log'; print('$',' '.join(map(str,command)),flush=True); env={**os.environ,**(extra_env or {})}
    with log.open('a',encoding='utf-8') as out:
        p=subprocess.Popen(list(map(str,command)),cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); out.write(line); out.flush()
        code=p.wait()
    if code: raise subprocess.CalledProcessError(code,command)
assert EXECUTE_API_CALLS,'Set EXECUTE_API_CALLS=True after checking ARM_ID and RUN_MODE'
collect=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset,'--variant-manifest',profile,'--config',config_path,'--run-dir',RUN_DIR,'--question-ids-file',IDS,'--chunks-path',chunks,'--bm25-path',sparse_index,'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--generator-model',GENERATOR_MODEL,'--prompt-id','p2','--system-prompt-file',prompt,'--context-depth',CONTEXT_DEPTH,'--source-retrievals',source_retrievals,'--generation-min-interval-seconds',GENERATOR_INTERVAL,'--max-attempts',3,'--retry-failed','--progress']
run(collect,'generate',{'GEMINI_API_KEY':GENERATOR_KEY})
run([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_prejudge')
judge=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',RUN_DIR,'--judge-provider','fireworks','--judge-model',JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',JUDGE_IDS,'--batch-size',1,'--max-workers',1,'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
run(judge,'judge',{'FIREWORKS_API_KEY':JUDGE_KEY})
run([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_final')

## 4. Review and export

In [ ]:
report=json.loads((RUN_DIR/'report.json').read_text()); predictions=rows(RUN_DIR/'predictions.jsonl'); judges=rows(RUN_DIR/'judge_results.jsonl')
generation_usage={key:sum(int((row.get('result') or {}).get('usage',{}).get(key,0)) for row in predictions if row.get('status')=='success') for key in ['input_tokens','output_tokens']}
judge_usage={key:sum(int(row.get('usage',{}).get(key,0)) for row in judges if row.get('status')=='success') for key in ['input_tokens','output_tokens']}
cost={'generation_usd':generation_usage['input_tokens']/1e6*GENERATOR_INPUT_USD_PER_MILLION+generation_usage['output_tokens']/1e6*GENERATOR_OUTPUT_USD_PER_MILLION,'judge_usd':judge_usage['input_tokens']/1e6*JUDGE_INPUT_USD_PER_MILLION+judge_usage['output_tokens']/1e6*JUDGE_OUTPUT_USD_PER_MILLION}; cost['total_usd']=cost['generation_usd']+cost['judge_usd']
summary={'arm':ARM_ID,'prompt_id':'p2','context_depth':CONTEXT_DEPTH,'run_mode':RUN_MODE,'questions':len(active_ids),'judge_questions':len(judge_ids),'coverage':report['coverage'],'qa':report.get('qa',{}),'citations':report.get('citations',{}),'ragas':report.get('ragas',{}),'latency':report.get('latency',{}),'usage':{'generation':generation_usage,'judge':judge_usage},'estimated_cost':cost,'source_retrieval_manifest_sha256':sha(RETRIEVAL/'run_manifest.json'),'repo_commit':REPO_COMMIT}
(RESULTS/'summary.json').write_text(json.dumps(summary,indent=2,sort_keys=True)+'\n'); display(pd.json_normalize(summary))
for name in ['run_manifest.json','report.json','report_summary.txt','predictions.jsonl','retrievals.jsonl','deterministic_scores.jsonl','judge_results.jsonl','attempts.jsonl','environment.json']:
    source=RUN_DIR/name
    if source.exists(): shutil.copy2(source,RESULTS/name)
bundle=Path(shutil.make_archive(str(WORK/f'phase2c_{ARM_ID}_p2_d{CONTEXT_DEPTH}_{RUN_MODE}_results'),'zip',root_dir=RESULTS)); destination=Path('/content/drive/MyDrive/newsqa_phase2c/results')/bundle.name; destination.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(bundle,destination)
print('Saved:',destination,'| SHA-256:',sha(destination))